# Dos bayesianas con etiquetas estimadas para julio

Base: `z495_python_lags_doble.ipynb`.
La primera BO utiliza las etiquetas reales de marzo a junio y predice julio.
La segunda incorpora julio al entrenamiento y optimiza también `prob_corte`;
su validación utiliza solamente etiquetas reales de marzo a junio.

# Librerías

In [1]:
import os
import cudf
import cupy as cp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd



from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
import optuna
from optuna.samplers import GPSampler
from optuna.trial import TrialState




print('cuDF:', cudf.__version__)
print('GPU:', cp.cuda.runtime.getDeviceProperties(0)['name'].decode())

cuDF: 26.08.01
GPU: NVIDIA GeForce RTX 3090


In [2]:
import lightgbm as lgb
import yaml
import rdata

In [3]:
# dispongo la carpeta con funciones e importo ternara. 
from pathlib import Path
import sys

# Obtiene la ruta del directorio actual y sube un nivel (.parent)
raiz_proyecto = Path().resolve().parent

# Agrega la ruta al sistema si no está ya incluida
if str(raiz_proyecto) not in sys.path:
  sys.path.append(str(raiz_proyecto))




# URLs

In [4]:
BASE_URL = os.path.join(
    os.path.expanduser("~"),
    "code", "DMEyF", "dmeyf2026"
)

DATA_FOLDER = os.path.join(BASE_URL, 'entregas', 'dataset')
DATA_URL = os.path.join(BASE_URL, "DATA")

EXP_URL = os.path.join(DATA_URL, "EXP")

DATA_TERNARIA_URL = os.path.join(DATA_FOLDER, 'competencia_01_ternaria.csv')
DATA_LAGS_URL = os.path.join(DATA_FOLDER, 'competencia_01_ternaria_lags.csv')
DATA_LAGS_FRAC_URL = os.path.join(DATA_FOLDER, 'competencia_01_ternaria_lf_k6_perdida4.csv')



carpeta_man_val = os.path.join(BASE_URL, "ensembles", "man_val")
archivo_validaciones = os.path.join(carpeta_man_val, "validaciones_manuales.csv")

experiment_name = "LAG-010"
experimento_segunda = "LAG-PSEUDO-001"
EXPERIMENT_URL = os.path.join(EXP_URL, experiment_name)

os.makedirs(EXPERIMENT_URL, exist_ok=True)

In [5]:
os.listdir(DATA_FOLDER)

['competencia_01_ternaria_lf_k6_perdida4.csv',
 'versiones_dataset_1.ipynb',
 'competencia_01_ternaria.csv',
 'competencia_01_crudo.csv',
 'competencia_01_ternaria_lags.csv']

# Constantes

In [6]:
SEED = 230047
future_segunda = [202108]

In [7]:
PARAM = {
    "experimento": experiment_name,
    "semilla_primigenia": SEED,
    "training_pct": 70,
    "train": [202103, 202104, 202105, 202106],
    "train_final": [202103, 202104, 202105, 202106],
    "future": [202107],
    "cortes": list(range(9_000, 15_001, 500)),
    "trainingstrategy": {"undersampling": 0.1},
    "hyperparametertuning": {
        "iteraciones": 200,
        "n_startup_trials": 60,
        "n_modelos_previos": 30,
        "xval_folds" : 3
    },
    "lgbm": {
        "param_fijos": {
            "boosting": "gbdt",
            "objective": "binary",
            # "metric": "auc",
            "metric": "average_precision",
            "feature_pre_filter": False,
            "first_metric_only": False,
            "boost_from_average": True,
            "force_row_wise": True,
            "deterministic": True,
            "verbosity": -100,
            "seed": SEED,
            "max_depth": -1,
            "min_gain_to_split": 0.0,
            "lambda_l1": 0.0,
            "lambda_l2": 0.0,
            "max_bin": 31,
            "bagging_fraction": 1.0,
            "pos_bagging_fraction": 1.0,
            "neg_bagging_fraction": 1.0,
            "is_unbalance": False,
            "scale_pos_weight": 1.0,
            "drop_rate": 0.1,
            "max_drop": 50,
            "skip_drop": 0.5,
            "extra_trees": False,
            "early_stopping": 0,
            "min_data_in_leaf": 0,
            "feature_fraction": 0.50,
            # "learning_rate": 0.005,
  
        }
    },
    "out": {"lgbm": {}},
}

# Lectura del dataset

In [8]:
# cudf no pandas -> GPU
df = cudf.read_csv(
    DATA_LAGS_FRAC_URL,
    dtype={'numero_de_cliente': 'int32', 'foto_mes': 'int32'}
)

In [9]:
df.head()

,numero_de_cliente,foto_mes,active_quarter,cliente_vip,internet,cliente_edad,cliente_antiguedad,mrentabilidad,mrentabilidad_annual,mcomisiones,...,Visa_madelantodolares_delta_lag_pond,Visa_fultimo_cierre_delta_lag_pond,Visa_mpagado_delta_lag_pond,Visa_mpagospesos_delta_lag_pond,Visa_mpagosdolares_delta_lag_pond,Visa_fechaalta_delta_lag_pond,Visa_mconsumototal_delta_lag_pond,Visa_cconsumos_delta_lag_pond,Visa_cadelantosefectivo_delta_lag_pond,Visa_mpagominimo_delta_lag_pond
0,12159854,202103,1,0,0,56,134,688.41,26701.84,98.82,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,12159854,202104,1,0,0,56,135,1622.57,27663.64,931.59,...,0.0,1.0,0.0,0.0,0.0,30.0,0.0,0.0,0.0,0.0
2,12159854,202105,1,0,0,56,136,1987.98,28738.37,934.07,...,0.0,3.375,0.0,1.818989404e-12,0.0,42.25,0.0,0.0,0.0,0.0
3,12159854,202106,1,0,0,56,137,2580.75,29494.76,915.19,...,0.0,-3.346938776,0.0,1.818989404e-12,0.0,50.69387755,0.0,0.0,0.0,0.0
4,12159854,202107,1,0,0,56,138,2686.85,29431.10,688.94,...,0.0,1.191176471,0.0,3.637978807e-12,0.0,58.39705882,0.0,0.0,0.0,0.0


In [10]:
cudf.crosstab(df['ternaria'].fillna('NA'), df['foto_mes'])

foto_mes,202103,202104,202105,202106,202107,202108
ternaria,,,,,,
BAJA+1,1019,964,1143,874,1103,0
BAJA+2,960,1139,870,1098,0,0
NA,0,0,0,0,163245,164647
continua,160921,161181,161755,162142,0,0


In [11]:
df_completo = df.copy()

# Optimización de hiperparámetros

Dos BO consecutivas con el mismo espacio de hiperparámetros.
La cantidad de corridas y de inicializaciones se configura en `PARAM`.
Se inicializan con modelos de `validaciones_manuales.csv`, priorizando el dataset utilizado;
las demás inicializaciones se muestrean del espacio.

## Preprocesamiento

In [12]:
dataset_train_cpu = (
    df_completo
    .loc[df_completo["foto_mes"].isin(PARAM["train"])]
    .to_pandas()
    .reset_index(drop=True)
)

# Partición estratificada 70/30, equivalente a fold 1 y fold 2 en z495.
indices_fold_1, indices_fold_2 = train_test_split(
    np.arange(len(dataset_train_cpu)),
    train_size=PARAM["training_pct"] / 100,
    stratify=dataset_train_cpu["ternaria"],
    random_state=PARAM["semilla_primigenia"],
)

dataset_train_cpu["fold"] = np.int8(2)
dataset_train_cpu.loc[indices_fold_1, "fold"] = np.int8(1)

# BAJA+1 y BAJA+2 forman la clase positiva, igual que en el R.
dataset_train_cpu["clase01"] = (
    dataset_train_cpu["ternaria"].isin(["BAJA+1", "BAJA+2"])
    .astype(np.int8)
)

rng = np.random.default_rng(PARAM["semilla_primigenia"])
dataset_train_cpu["azar"] = rng.random(len(dataset_train_cpu))
dataset_train_cpu["training"] = (
    dataset_train_cpu["fold"].eq(1)
    & (
        (dataset_train_cpu["azar"] <= PARAM["trainingstrategy"]["undersampling"])
        | dataset_train_cpu["ternaria"].isin(["BAJA+1", "BAJA+2"])
    )
).astype(np.int8)

columnas_excluidas = {
    "ternaria",
    "ternaria_lag1",
    "clase01",
    "fold",
    "azar",
    "training",
    "cprestamos_personales",
    "cprestamos_personales_lag1",
    "cprestamos_personales_delta1",
    "mprestamos_personales",
    "mprestamos_personales_lag1",
    "mprestamos_personales_delta1"
}

campos_buenos = [
    columna
    for columna in dataset_train_cpu.columns
    if columna not in columnas_excluidas
]

mask_training = dataset_train_cpu["training"].eq(1)
X_train_bo = dataset_train_cpu.loc[mask_training, campos_buenos]
y_train_bo = dataset_train_cpu.loc[mask_training, "clase01"]

# Replica literalmente z495: validation contiene todas las filas con training == 0.
mask_validation = dataset_train_cpu["training"].eq(0)
X_validate_bo = dataset_train_cpu.loc[mask_validation, campos_buenos]
y_validate_bo = dataset_train_cpu.loc[mask_validation, "clase01"]

dtrain = lgb.Dataset(
    data=X_train_bo,
    label=y_train_bo,
    feature_name=campos_buenos,
    free_raw_data=False,
)

dvalidate = lgb.Dataset(
    data=X_validate_bo,
    label=y_validate_bo,
    feature_name=campos_buenos,
    free_raw_data=False,
)

print("Filas para training:", len(X_train_bo))
print("Filas para validation:", len(X_validate_bo))
print("Columnas:", len(campos_buenos))
dataset_train_cpu.groupby(["fold", "ternaria", "training"]).size()

Filas para training: 50866
Filas para validation: 603200
Columnas: 456


fold  ternaria  training
1     BAJA+1    1             2800
      BAJA+2    1             2847
      continua  0           406980
                1            45219
2     BAJA+1    0             1200
      BAJA+2    0             1220
      continua  0           193800
dtype: int64

## Función objetivo: Average Precision sobre validation

In [13]:
def estimar_auc_lightgbm(parametros_variables):
    parametros_completos = {
        **PARAM["lgbm"]["param_fijos"],
        **parametros_variables,
    }

    # En Python conviene pasar el número de árboles por su argumento dedicado.
    num_boost_round = int(parametros_completos.pop("num_iterations"))

    resultados_cv = lgb.cv(
        params=parametros_completos,
        train_set=dtrain,
        num_boost_round=num_boost_round,
        nfold=PARAM["hyperparametertuning"]["xval_folds"],
        stratified=True,
        shuffle=True,
        seed=PARAM["semilla_primigenia"],
    )

    clave_auc = next(
        clave for clave in resultados_cv
        if clave.endswith("auc-mean")
    )

    # Sin early stopping, equivale al score al finalizar num_iterations.
    return float(resultados_cv[clave_auc][-1])

In [14]:
def estimar_ap_lightgbm(parametros_variables):
    parametros_variables = parametros_variables.copy()
    prob_corte = parametros_variables.pop("prob_corte", None)

    if prob_corte is None:
        dtrain_bo = dtrain
    else:
        clase_julio = np.isin(etiquetar_julio(prob_corte), ["BAJA+1", "BAJA+2"])
        training_julio = (
            (azar_julio <= PARAM["trainingstrategy"]["undersampling"])
            | clase_julio
        )
        dtrain_bo = lgb.Dataset(
            data=pd.concat(
                [X_train_bo, dataset_julio_cpu.loc[training_julio, campos_buenos]],
                ignore_index=True,
            ),
            label=np.concatenate([y_train_bo.to_numpy(), clase_julio[training_julio]]),
            feature_name=campos_buenos,
        )

    parametros_completos = {
        **PARAM["lgbm"]["param_fijos"],
        **parametros_variables,
    }

    num_boost_round = int(parametros_completos.pop("num_iterations"))

    modelo = lgb.train(
        params=parametros_completos,
        train_set=dtrain_bo,
        num_boost_round=num_boost_round,
        valid_sets=[dvalidate],
        valid_names=["val"],
    )

    return float(modelo.best_score["val"]["average_precision"])

In [15]:
ganancia_validate_bo = np.where(
    dataset_train_cpu.loc[mask_validation, "ternaria"].eq("BAJA+2"),
    1_072_500,
    -27_500,
)


def estimar_ganancia_lightgbm(parametros_variables):
    parametros_completos = {
        **PARAM["lgbm"]["param_fijos"],
        **parametros_variables,
    }

    num_boost_round = int(
        parametros_completos.pop("num_iterations")
    )

    modelo = lgb.train(
        params=parametros_completos,
        train_set=dtrain,
        num_boost_round=num_boost_round,
    )

    prediccion = modelo.predict(X_validate_bo)
    orden = np.argsort(-prediccion)

    ganancia_acumulada = np.cumsum(
        ganancia_validate_bo[orden]
    )

    return float(ganancia_acumulada.max())

In [16]:
rangos_bo = {'num_iterations': (400, 4000),
 'num_leaves': (16, 2048),
 'feature_fraction': (0.1, 0.75),
 'min_sum_hessian_in_leaf': (0.0001, 0.1),
 'learning_rate': (0.002, 0.025),
 'min_data_in_leaf': (0, 1000),
 'scale_pos_weight': (0.5, 1.0)}

def objective(trial):
    parametros_variables = {
        "num_iterations": trial.suggest_int(
            "num_iterations", *rangos_bo["num_iterations"]
        ),
        "num_leaves": trial.suggest_int(
            "num_leaves", *rangos_bo["num_leaves"], log=True
        ),
        "feature_fraction": trial.suggest_float(
            "feature_fraction", *rangos_bo["feature_fraction"]
        ),
        "min_sum_hessian_in_leaf": trial.suggest_float(
            "min_sum_hessian_in_leaf",
            *rangos_bo["min_sum_hessian_in_leaf"],
            log=True,
        ),
        "learning_rate": trial.suggest_float(
            "learning_rate", *rangos_bo["learning_rate"], log=True
        ),
        "min_data_in_leaf": trial.suggest_int(
            "min_data_in_leaf",
            *rangos_bo["min_data_in_leaf"],
            log=rangos_bo["min_data_in_leaf"][0] > 0,
        ),
    }
    if "scale_pos_weight" in rangos_bo:
        parametros_variables["scale_pos_weight"] = trial.suggest_float(
            "scale_pos_weight", *rangos_bo["scale_pos_weight"]
        )
    if "prob_corte" in rangos_bo:
        parametros_variables["prob_corte"] = trial.suggest_float(
            "prob_corte", *rangos_bo["prob_corte"]
        )
    return estimar_ap_lightgbm(parametros_variables)
    # return estimar_auc_lightgbm(parametros_variables)
    # return estimar_ganancia_lightgbm(parametros_variables)

## Modelos para las inicializaciones

In [17]:
# Misma conversión de RDS usada en la validación manual Python.
def convertir_r(valor):
    if isinstance(valor, dict):
        return {k: convertir_r(v) for k, v in valor.items()}
    if isinstance(valor, np.ndarray):
        if valor.size == 1:
            return convertir_r(valor.item())
        return [convertir_r(v) for v in valor.tolist()]
    if isinstance(valor, (list, tuple)):
        return [convertir_r(v) for v in valor]
    if isinstance(valor, np.generic):
        return convertir_r(valor.item())
    if isinstance(valor, float) and np.isfinite(valor) and valor.is_integer():
        return int(valor)
    return valor

validaciones = pd.read_csv(archivo_validaciones)
validaciones = validaciones.sort_values("ganancia", ascending=False)
dataset_file = os.path.basename(DATA_LAGS_FRAC_URL)
es_dataset_actual = validaciones["dataset"].eq(dataset_file)
modelos_iniciales = pd.concat([
    validaciones.loc[es_dataset_actual],
    validaciones.loc[~es_dataset_actual],
]).head(PARAM["hyperparametertuning"]["n_modelos_previos"])

puntos_iniciales_bo = []
for id_validacion in modelos_iniciales["id_validacion"]:
    parametros = convertir_r(rdata.read_rds(
        os.path.join(carpeta_man_val, f"{int(id_validacion)}.rds")
    ))
    if "num_iterations" not in parametros:
        parametros = next(iter(parametros.values()))
    parametros = parametros.copy()
    # El RDS guarda este parámetro ajustado para entrenar sin undersampling.
    parametros["min_data_in_leaf"] = int(np.rint(
        parametros["min_data_in_leaf"] * PARAM["trainingstrategy"]["undersampling"]
    ))
    puntos_iniciales_bo.append({
        nombre: parametros.get(nombre, PARAM["lgbm"]["param_fijos"].get(nombre))
        for nombre in rangos_bo
    })

modelos_iniciales

,id_validacion,ganancia,seeds,dataset,cantidad_registros_totales,cantidad_registros_entrenamiento,cantidad_registros_validacion,meses_entrenamiento,meses_validacion,columnas_excluidas,benchmark_original
50,51,474831500,"230047,230059,230063,230077,230081",competencia_01_ternaria_lf_k6_perdida4.csv,983061,326184,164114,"202103,202104",202106,"ternaria,ternaria_lag1,clase01,fold,azar,train...",benchmark_multisemilla_lags20260924_090149
43,44,474831500,"230047,230059,230063,230077,230081",competencia_01_ternaria_lf_k6_perdida4.csv,983061,326184,164114,"202103,202104",202106,"ternaria,ternaria_lag1,clase01,fold,azar,train...",benchmark_multisemilla_lags20260924_090149
38,39,474831500,"230047,230059,230063,230077,230081",competencia_01_ternaria_lf_k6_perdida4.csv,983061,326184,164114,"202103,202104",202106,"ternaria,ternaria_lag1,clase01,fold,azar,train...",benchmark_multisemilla_lags20260924_123618
31,32,473517000,"230047,230059,230063,230077,230081",competencia_01_ternaria_lf_k6_perdida4.csv,983061,326184,164114,"202103,202104",202106,"ternaria,ternaria_lag1,clase01,fold,azar,train...",benchmark_multisemilla_lags20260924_090643
42,43,473517000,"230047,230059,230063,230077,230081",competencia_01_ternaria_lf_k6_perdida4.csv,983061,326184,164114,"202103,202104",202106,"ternaria,ternaria_lag1,clase01,fold,azar,train...",benchmark_multisemilla_lags20260924_090643
27,28,473357500,NaN,competencia_01_ternaria_lf_k6_perdida4.csv,983061,326184,164114,"202103,202104",202106,"ternaria,ternaria_lag1,clase01,fold,azar,train...",benchmark_multisemilla_lags20260924_090149
28,29,472230000,NaN,competencia_01_ternaria_lf_k6_perdida4.csv,983061,326184,164114,"202103,202104",202106,"ternaria,ternaria_lag1,clase01,fold,azar,train...",benchmark_multisemilla_lags20260924_090643
47,48,465883000,"230047,230059,230063,230077,230081",competencia_01_ternaria_lf_k6_perdida4.csv,983061,326184,164114,"202103,202104",202106,"ternaria,ternaria_lag1,clase01,fold,azar,train...",benchmark_multisemilla_lags20260924_084709
34,35,465883000,"230047,230059,230063,230077,230081",competencia_01_ternaria_lf_k6_perdida4.csv,983061,326184,164114,"202103,202104",202106,"ternaria,ternaria_lag1,clase01,fold,azar,train...",benchmark_multisemilla_lags20260924_132722
32,33,465883000,"230047,230059,230063,230077,230081",competencia_01_ternaria_lf_k6_perdida4.csv,983061,326184,164114,"202103,202104",202106,"ternaria,ternaria_lag1,clase01,fold,azar,train...",benchmark_multisemilla_lags20260924_132722


## Estudio Optuna y reanudación

In [18]:
n_startup_trials = PARAM["hyperparametertuning"]["n_startup_trials"]
n_trials_total = PARAM["hyperparametertuning"]["iteraciones"]

sampler = GPSampler(
    seed=PARAM["semilla_primigenia"],
    n_startup_trials=n_startup_trials,
    deterministic_objective=False,
)

study_db_url = os.path.join(EXPERIMENT_URL, "optuna_study.db")
storage_url = f"sqlite:///{study_db_url}"

study = optuna.create_study(
    study_name=experiment_name,
    direction="maximize",
    sampler=sampler,
    storage=storage_url,
    load_if_exists=True,
)

# Se encolan solamente al crear el estudio; conserva también HP repetidos.
if not study.trials:
    for punto in puntos_iniciales_bo:
        study.enqueue_trial(punto)

/tmp/ipykernel_397057/96132893.py:4: ExperimentalWarning: GPSampler is experimental (supported from v3.6.0). The interface can change in the future.
  sampler = GPSampler(
[I 2026-10-05 23:27:32,088] A new study created in RDB with name: LAG-010


In [19]:
trials_completos = sum(
    trial.state == TrialState.COMPLETE
    for trial in study.trials
)
trials_restantes = max(0, n_trials_total - trials_completos)

print("Trials completos:", trials_completos)
print("Trials restantes:", trials_restantes)

if trials_restantes > 0:
    study.optimize(
        objective,
        n_trials=trials_restantes,
        show_progress_bar=True,
    )
else:
    print("c'est fini")

Trials completos: 0
Trials restantes: 200


  0%|          | 0/200 [00:00<?, ?it/s]

[I 2026-10-05 23:27:58,576] Trial 0 finished with value: 0.12584029556988666 and parameters: {'num_iterations': 400, 'num_leaves': 256, 'feature_fraction': 0.239049018582973, 'min_sum_hessian_in_leaf': 0.1, 'learning_rate': 0.0152796385349906, 'min_data_in_leaf': 137, 'scale_pos_weight': 0.5}. Best is trial 0 with value: 0.12584029556988666.
[I 2026-10-05 23:28:15,885] Trial 1 finished with value: 0.12584029556988666 and parameters: {'num_iterations': 400, 'num_leaves': 256, 'feature_fraction': 0.239049018582973, 'min_sum_hessian_in_leaf': 0.1, 'learning_rate': 0.0152796385349906, 'min_data_in_leaf': 137, 'scale_pos_weight': 0.5}. Best is trial 0 with value: 0.12584029556988666.
[I 2026-10-05 23:28:32,624] Trial 2 finished with value: 0.12584029556988666 and parameters: {'num_iterations': 400, 'num_leaves': 256, 'feature_fraction': 0.239049018582973, 'min_sum_hessian_in_leaf': 0.1, 'learning_rate': 0.0152796385349906, 'min_data_in_leaf': 137, 'scale_pos_weight': 0.5}. Best is trial 0 w

/home/marco/anaconda3/envs/cmeyf-cuda13/lib/python3.11/site-packages/optuna/trial/_trial.py:664: UserWarning: Fixed parameter min_sum_hessian_in_leaf with value 1e-06 is out of range for distribution FloatDistribution(high=0.1, log=True, low=0.0001, step=None).
  optuna_warn(


[I 2026-10-05 23:38:27,226] Trial 16 finished with value: 0.11197237478493562 and parameters: {'num_iterations': 2152, 'num_leaves': 16, 'feature_fraction': 0.4, 'min_sum_hessian_in_leaf': 1e-06, 'learning_rate': 0.004, 'min_data_in_leaf': 0, 'scale_pos_weight': 1}. Best is trial 3 with value: 0.12731982780231527.
[I 2026-10-05 23:39:36,283] Trial 17 finished with value: 0.11431038175128952 and parameters: {'num_iterations': 2837, 'num_leaves': 16, 'feature_fraction': 0.4, 'min_sum_hessian_in_leaf': 1e-06, 'learning_rate': 0.004, 'min_data_in_leaf': 0, 'scale_pos_weight': 1}. Best is trial 3 with value: 0.12731982780231527.
[I 2026-10-05 23:39:55,440] Trial 18 finished with value: 0.12684003672548586 and parameters: {'num_iterations': 712, 'num_leaves': 86, 'feature_fraction': 0.4, 'min_sum_hessian_in_leaf': 1e-06, 'learning_rate': 0.00757969257611117, 'min_data_in_leaf': 0, 'scale_pos_weight': 1}. Best is trial 3 with value: 0.12731982780231527.
[I 2026-10-05 23:40:09,086] Trial 19 fi

## Resultados de la optimización

In [20]:
tb_bayesiana = study.trials_dataframe()
tb_bayesiana["iter"] = np.arange(1, len(tb_bayesiana) + 1)
tb_bayesiana = (
    tb_bayesiana
    .sort_values("value", ascending=False, na_position="last")
    .reset_index(drop=True)
)
tb_bayesiana.to_csv(
    os.path.join(EXPERIMENT_URL, "BO_log.txt"),
    sep="	",
    index=False,
)

PARAM["out"]["lgbm"]["mejores_hiperparametros"] = dict(study.best_trial.params)
PARAM["out"]["lgbm"]["y"] = float(study.best_value)

with open(os.path.join(EXPERIMENT_URL, "PARAM.yml"), "w", encoding="utf-8") as archivo:
    yaml.safe_dump(PARAM, archivo, sort_keys=False, allow_unicode=True)

print(PARAM["out"]["lgbm"]["mejores_hiperparametros"])
print("Mejor Average Precision:", PARAM["out"]["lgbm"]["y"])
tb_bayesiana.head()

{'num_iterations': 4000, 'num_leaves': 166, 'feature_fraction': 0.27641343005495034, 'min_sum_hessian_in_leaf': 0.1, 'learning_rate': 0.011685329568129043, 'min_data_in_leaf': 0, 'scale_pos_weight': 1.0}
Mejor Average Precision: 0.14422319332138248


,number,value,datetime_start,datetime_complete,duration,params_feature_fraction,params_learning_rate,params_min_data_in_leaf,params_min_sum_hessian_in_leaf,params_num_iterations,params_num_leaves,params_scale_pos_weight,system_attrs_fixed_params,system_attrs_gp:relative_params:0,state,iter
0,159,0.144223,2026-10-06 02:53:54.336657,2026-10-06 02:55:52.810355,0 days 00:01:58.473698,0.276413,0.011685,0,0.100000,4000,166,1.0,NaN,"{""feature_fraction"": 0.27641343005495034, ""lea...",COMPLETE,160
1,189,0.143950,2026-10-06 03:49:03.750681,2026-10-06 03:50:59.960659,0 days 00:01:56.209978,0.272505,0.011555,0,0.009703,4000,157,1.0,NaN,"{""feature_fraction"": 0.272504623324586, ""learn...",COMPLETE,190
2,100,0.143589,2026-10-06 01:08:00.134036,2026-10-06 01:09:15.912086,0 days 00:01:15.778050,0.363927,0.025000,0,0.006716,2472,158,1.0,NaN,"{""feature_fraction"": 0.36392719749807123, ""lea...",COMPLETE,101
3,192,0.143466,2026-10-06 03:54:22.107306,2026-10-06 03:56:22.735967,0 days 00:02:00.628661,0.308348,0.011828,0,0.044820,4000,167,1.0,NaN,"{""feature_fraction"": 0.3083476155787994, ""lear...",COMPLETE,193
4,191,0.143376,2026-10-06 03:52:27.974870,2026-10-06 03:54:22.075156,0 days 00:01:54.100286,0.253460,0.011467,0,0.075461,4000,154,1.0,NaN,"{""feature_fraction"": 0.25346023723886746, ""lea...",COMPLETE,192


# Producción

## Final training sobre todos los meses de `PARAM["train_final"]`

In [21]:
dataset_final_cpu = (
    df_completo
    .loc[df_completo["foto_mes"].isin(PARAM["train_final"])]
    .to_pandas()
    .reset_index(drop=True)
)
dataset_final_cpu["clase01"] = (
    dataset_final_cpu["ternaria"].isin(["BAJA+1", "BAJA+2"])
    .astype(np.int8)
)

dataset_final_cpu.groupby("ternaria").size()

ternaria
BAJA+1        4000
BAJA+2        4067
continua    645999
dtype: int64

In [22]:
dataset_final_cpu.clase01.value_counts()

clase01
0    645999
1      8067
Name: count, dtype: int64

In [23]:
param_final = {
    **PARAM["lgbm"]["param_fijos"],
    **PARAM["out"]["lgbm"]["mejores_hiperparametros"],
}

num_boost_round_final = int(param_final.pop("num_iterations"))
param_final["min_data_in_leaf"] = round(
    param_final["min_data_in_leaf"] / PARAM["trainingstrategy"]["undersampling"]
)
param_final

{'boosting': 'gbdt',
 'objective': 'binary',
 'metric': 'average_precision',
 'feature_pre_filter': False,
 'first_metric_only': False,
 'boost_from_average': True,
 'force_row_wise': True,
 'deterministic': True,
 'verbosity': -100,
 'seed': 230047,
 'max_depth': -1,
 'min_gain_to_split': 0.0,
 'lambda_l1': 0.0,
 'lambda_l2': 0.0,
 'max_bin': 31,
 'bagging_fraction': 1.0,
 'pos_bagging_fraction': 1.0,
 'neg_bagging_fraction': 1.0,
 'is_unbalance': False,
 'scale_pos_weight': 1.0,
 'drop_rate': 0.1,
 'max_drop': 50,
 'skip_drop': 0.5,
 'extra_trees': False,
 'early_stopping': 0,
 'min_data_in_leaf': 0,
 'feature_fraction': 0.27641343005495034,
 'num_leaves': 166,
 'min_sum_hessian_in_leaf': 0.1,
 'learning_rate': 0.011685329568129043}

In [24]:
print(campos_buenos)

['numero_de_cliente', 'foto_mes', 'active_quarter', 'cliente_vip', 'internet', 'cliente_edad', 'cliente_antiguedad', 'mrentabilidad', 'mrentabilidad_annual', 'mcomisiones', 'mactivos_margen', 'mpasivos_margen', 'cproductos', 'tcuentas', 'ccuenta_corriente', 'mcuenta_corriente_adicional', 'mcuenta_corriente', 'ccaja_ahorro', 'mcaja_ahorro', 'mcaja_ahorro_adicional', 'mcaja_ahorro_dolares', 'cdescubierto_preacordado', 'mcuentas_saldo', 'ctarjeta_debito', 'ctarjeta_debito_transacciones', 'mautoservicio', 'ctarjeta_visa', 'ctarjeta_visa_transacciones', 'mtarjeta_visa_consumo', 'ctarjeta_master', 'ctarjeta_master_transacciones', 'mtarjeta_master_consumo', 'cprestamos_prendarios', 'mprestamos_prendarios', 'cprestamos_hipotecarios', 'mprestamos_hipotecarios', 'cplazo_fijo', 'mplazo_fijo_dolares', 'mplazo_fijo_pesos', 'cinversion1', 'minversion1_pesos', 'minversion1_dolares', 'cinversion2', 'minversion2', 'cseguro_vida', 'cseguro_auto', 'cseguro_vivienda', 'cseguro_accidentes_personales', 'cca

In [25]:
dtrain_final = lgb.Dataset(
    data=dataset_final_cpu[campos_buenos],
    label=dataset_final_cpu["clase01"],
    feature_name=campos_buenos,
)

modelo_final = lgb.train(
    params=param_final,
    train_set=dtrain_final,
    num_boost_round=num_boost_round_final,
)

## Importancia de variables y modelo

In [26]:
arboles_df = modelo_final.trees_to_dataframe()
splits_df = arboles_df.loc[arboles_df["split_feature"].notna()]

tb_importancia = (
    splits_df
    .groupby("split_feature", as_index=False)
    .agg(
        Gain=("split_gain", "sum"),
        Cover=("count", "sum"),
        Frequency=("split_feature", "size"),
    )
    .rename(columns={"split_feature": "Feature"})
)

for columna in ["Gain", "Cover", "Frequency"]:
    total = tb_importancia[columna].sum()
    if total > 0:
        tb_importancia[columna] = tb_importancia[columna] / total

tb_importancia = tb_importancia.sort_values("Gain", ascending=False)
tb_importancia.to_csv(
    os.path.join(EXPERIMENT_URL, "impo.txt"),
    sep="\t",
    index=False,
)
modelo_final.save_model(os.path.join(EXPERIMENT_URL, "modelo.txt"))

tb_importancia.head(20)

,Feature,Gain,Cover,Frequency
302,ctrx_quarter,0.044971,0.008782,0.009623
321,mcaja_ahorro,0.038035,0.006863,0.009017
393,mpayroll,0.025125,0.012698,0.002436
363,mcuentas_saldo,0.024996,0.012685,0.011276
425,mtarjeta_visa_consumo,0.024950,0.003898,0.004353
237,cpayroll_trx,0.024669,0.006266,0.001368
293,ctarjeta_visa_transacciones,0.017842,0.004333,0.005036
414,mrentabilidad_annual,0.015582,0.017496,0.015442
354,mcuenta_corriente,0.015533,0.009716,0.013152
269,ctarjeta_debito_transacciones,0.014901,0.002272,0.003826


## Scoring de julio

In [27]:
df_completo.shape

(983061, 459)

In [28]:
dfuture_cpu = (
    df_completo
    .loc[df_completo["foto_mes"].isin(PARAM["future"])]
    .to_pandas()
    .reset_index(drop=True)
)

prediccion = modelo_final.predict(dfuture_cpu[campos_buenos])

tb_prediccion = dfuture_cpu[["numero_de_cliente", "foto_mes", "ternaria"]].copy()
tb_prediccion["prob"] = prediccion
# Estas probabilidades se guardan en la carpeta de la segunda BO.


tb_prediccion.head()

,numero_de_cliente,foto_mes,ternaria,prob
0,12159854,202107,NaN,0.000115
1,12159858,202107,NaN,0.000417
2,12160484,202107,NaN,0.000160
3,12160591,202107,NaN,0.000007
4,12160747,202107,NaN,0.000388


## Documentación de la primera BO

In [29]:
contenido_md = [
    f"# Resumen — {experiment_name}",
    "",
    "- Etiquetas de entrenamiento y validación: reales.",
    "- Validación: todas las filas de marzo a junio con training == 0, como en la notebook base.",
    "- Julio no tiene BAJA+2 real: no se calcula ganancia sobre ese mes.",
    f"- Registros de entrenamiento BO: {len(X_train_bo)}",
    f"- Registros de validación BO: {len(X_validate_bo)}",
    f"- Registros de entrenamiento final: {len(dataset_final_cpu)}",
    f"- Registros predichos: {len(dfuture_cpu)}",
    f"- Probabilidades de julio: se guardan en `{experimento_segunda}/predicciones_julio/prediccion.txt`.",
    f"- Dataset: `{dataset_file}`",
    f"- Meses de entrenamiento: {', '.join(map(str, PARAM['train']))}",
    f"- Meses de entrenamiento final: {', '.join(map(str, PARAM['train_final']))}",
    f"- Meses a predecir: {', '.join(map(str, PARAM['future']))}",
    f"- Semilla: {PARAM['semilla_primigenia']}",
    "- Clase positiva: BAJA+1 y BAJA+2.",
    f"- Columnas excluidas: {', '.join(sorted(columnas_excluidas))}",
    f"- Porcentaje para entrenamiento en el split: {PARAM['training_pct']}",
    f"- Undersampling: {PARAM['trainingstrategy']['undersampling']}",
    f"- Corridas por BO: {PARAM['hyperparametertuning']['iteraciones']}",
    f"- Inicializaciones totales: {PARAM['hyperparametertuning']['n_startup_trials']}",
    f"- Inicializaciones desde RDS: {len(puntos_iniciales_bo)}",
    f"- Inicializaciones aleatorias: {PARAM['hyperparametertuning']['n_startup_trials'] - len(puntos_iniciales_bo)}",
    f"- Fuente de las inicializaciones: `{archivo_validaciones}`",
    f"- IDs de validación usados: {', '.join(map(str, modelos_iniciales['id_validacion']))}",
    "- Selección: mayor ganancia dentro del dataset actual; se completa con los demás datasets.",
    "- min_data_in_leaf de los RDS: se revierte el ajuste por undersampling antes de inicializar.",
    f"- Rangos de HP: `{rangos_bo}`",
    f"- Mejor trial de Optuna: {study.best_trial.number}",
    f"- Mejor Average Precision: {PARAM['out']['lgbm']['y']}",
    f"- Mejores HP: `{PARAM['out']['lgbm']['mejores_hiperparametros']}`",
]
with open(os.path.join(EXPERIMENT_URL, "resumen.md"), "w", encoding="utf-8") as archivo:
    archivo.write("\n".join(contenido_md) + "\n")

## Segunda BO: incorpora julio al entrenamiento y optimiza `prob_corte`

In [30]:
experimento_origen = experiment_name
meses_reales = PARAM["train"].copy()
meses_julio = PARAM["future"].copy()
dataset_julio_cpu = dfuture_cpu.copy()
prob_julio = prediccion.copy()
baja1_julio = dataset_julio_cpu["ternaria"].isin(["BAJA+1"]).to_numpy()

# Los BAJA+1 reales se conservan; el corte asigna BAJA+2 al resto.
def etiquetar_julio(prob_corte):
    return np.where(
        baja1_julio,
        "BAJA+1",
        np.where(prob_julio >= prob_corte, "BAJA+2", "CONTINUA"),
    )

# El rango se obtiene de los puestos definidos por PARAM["cortes"].
probabilidades_ordenadas = np.sort(prob_julio[~baja1_julio])[::-1]
rangos_bo["prob_corte"] = (
    float(probabilidades_ordenadas[max(PARAM["cortes"]) - 1]),
    float(probabilidades_ordenadas[min(PARAM["cortes"]) - 1]),
)

experiment_name = experimento_segunda
EXPERIMENT_URL = os.path.join(EXP_URL, experiment_name)
os.makedirs(EXPERIMENT_URL, exist_ok=True)
PARAM["experimento"] = experiment_name
PARAM["train"] = meses_reales + meses_julio
PARAM["train_final"] = PARAM["train"].copy()
PARAM["future"] = future_segunda.copy()
PARAM["out"]["lgbm"] = {}

carpeta_predicciones_julio = os.path.join(EXPERIMENT_URL, "predicciones_julio")
os.makedirs(carpeta_predicciones_julio, exist_ok=True)
dataset_julio_cpu[["numero_de_cliente", "foto_mes"]].assign(prob=prob_julio).to_csv(
    os.path.join(carpeta_predicciones_julio, "prediccion.txt"),
    sep="\t",
    index=False,
)

# Se mantiene la partición real de la primera BO. Julio no entra en dvalidate.
# El azar de undersampling de julio permanece fijo al variar prob_corte.
rng = np.random.default_rng(PARAM["semilla_primigenia"])
azar_julio = rng.random(len(dataset_julio_cpu))

print("Rango de prob_corte:", rangos_bo["prob_corte"])
print("Meses de validación:", meses_reales)

Rango de prob_corte: (0.0035446538114082874, 0.008646300221991301)
Meses de validación: [202103, 202104, 202105, 202106]


## Estudio Optuna y reanudación

In [31]:
n_startup_trials = PARAM["hyperparametertuning"]["n_startup_trials"]
n_trials_total = PARAM["hyperparametertuning"]["iteraciones"]

sampler = GPSampler(
    seed=PARAM["semilla_primigenia"],
    n_startup_trials=n_startup_trials,
    deterministic_objective=False,
)

study_db_url = os.path.join(EXPERIMENT_URL, "optuna_study.db")
storage_url = f"sqlite:///{study_db_url}"

study = optuna.create_study(
    study_name=experiment_name,
    direction="maximize",
    sampler=sampler,
    storage=storage_url,
    load_if_exists=True,
)

# Se encolan solamente al crear el estudio; conserva también HP repetidos.
if not study.trials:
    for punto in puntos_iniciales_bo:
        study.enqueue_trial(punto)

/tmp/ipykernel_397057/96132893.py:4: ExperimentalWarning: GPSampler is experimental (supported from v3.6.0). The interface can change in the future.
  sampler = GPSampler(
[I 2026-10-06 04:17:51,229] A new study created in RDB with name: LAG-PSEUDO-001


In [32]:
trials_completos = sum(
    trial.state == TrialState.COMPLETE
    for trial in study.trials
)
trials_restantes = max(0, n_trials_total - trials_completos)

print("Trials completos:", trials_completos)
print("Trials restantes:", trials_restantes)

if trials_restantes > 0:
    study.optimize(
        objective,
        n_trials=trials_restantes,
        show_progress_bar=True,
    )
else:
    print("c'est fini")

Trials completos: 0
Trials restantes: 200


  0%|          | 0/200 [00:00<?, ?it/s]

[I 2026-10-06 04:18:09,640] Trial 0 finished with value: 0.11968225466366741 and parameters: {'num_iterations': 400, 'num_leaves': 256, 'feature_fraction': 0.239049018582973, 'min_sum_hessian_in_leaf': 0.1, 'learning_rate': 0.0152796385349906, 'min_data_in_leaf': 137, 'scale_pos_weight': 0.5, 'prob_corte': 0.004017753422711737}. Best is trial 0 with value: 0.11968225466366741.
[I 2026-10-06 04:18:26,994] Trial 1 finished with value: 0.12165313532584009 and parameters: {'num_iterations': 400, 'num_leaves': 256, 'feature_fraction': 0.239049018582973, 'min_sum_hessian_in_leaf': 0.1, 'learning_rate': 0.0152796385349906, 'min_data_in_leaf': 137, 'scale_pos_weight': 0.5, 'prob_corte': 0.0058642247881834355}. Best is trial 1 with value: 0.12165313532584009.
[I 2026-10-06 04:18:44,316] Trial 2 finished with value: 0.12164201981796191 and parameters: {'num_iterations': 400, 'num_leaves': 256, 'feature_fraction': 0.239049018582973, 'min_sum_hessian_in_leaf': 0.1, 'learning_rate': 0.0152796385349

/home/marco/anaconda3/envs/cmeyf-cuda13/lib/python3.11/site-packages/optuna/trial/_trial.py:664: UserWarning: Fixed parameter min_sum_hessian_in_leaf with value 1e-06 is out of range for distribution FloatDistribution(high=0.1, log=True, low=0.0001, step=None).
  optuna_warn(


[I 2026-10-06 04:28:31,100] Trial 16 finished with value: 0.10439645228040688 and parameters: {'num_iterations': 2152, 'num_leaves': 16, 'feature_fraction': 0.4, 'min_sum_hessian_in_leaf': 1e-06, 'learning_rate': 0.004, 'min_data_in_leaf': 0, 'scale_pos_weight': 1, 'prob_corte': 0.006980912410206179}. Best is trial 11 with value: 0.1269863270962039.


...

[I 2026-10-06 04:30:14,037] Trial 18 finished with value: 0.11931078435342186 and parameters: {'num_iterations': 712, 'num_leaves': 86, 'feature_fraction': 0.4, 'min_sum_hessian_in_leaf': 1e-06, 'learning_rate': 0.00757969257611117, 'min_data_in_leaf': 0, 'scale_pos_weight': 1, 'prob_corte': 0.006810223898698845}. Best is trial 11 with value: 0.1269863270962039.
[I 2026-10-06 04:30:33,536] Trial 19 finished with value: 0.1005513414973785 and parameters: {'num_iterations': 611, 'num_leaves': 20, 'feature_fraction': 0.552791861343054, 'min_sum_hessian_in_leaf': 0.0377107584841935, 'learning_rate': 0.01, 'min_data_in_leaf': 0, 'scale_pos_weight': 1, 'prob_corte': 0.004731371489105197}. Best is trial 11 with value: 0.1269863270962039.
[I 2026-10-06 04:31:01,500] Trial 20 finished with value: 0.12429003831774788 and parameters: {'num_iterations': 715, 'num_leaves': 164, 'feature_fraction': 0.4, 'min_sum_hessian_in_leaf': 0.1, 'learning_rate': 0.006, 'min_data_in_leaf': 0, 'scale_pos_weight'

/home/marco/anaconda3/envs/cmeyf-cuda13/lib/python3.11/site-packages/optuna/trial/_trial.py:664: UserWarning: Fixed parameter min_sum_hessian_in_leaf with value 1e-06 is out of range for distribution FloatDistribution(high=0.1, log=True, low=0.0001, step=None).
  optuna_warn(


[I 2026-10-06 04:31:26,668] Trial 21 finished with value: 0.11838984982207891 and parameters: {'num_iterations': 712, 'num_leaves': 86, 'feature_fraction': 0.4, 'min_sum_hessian_in_leaf': 1e-06, 'learning_rate': 0.00757969257611117, 'min_data_in_leaf': 0, 'scale_pos_weight': 1, 'prob_corte': 0.005167078984177872}. Best is trial 11 with value: 0.1269863270962039.


/home/marco/anaconda3/envs/cmeyf-cuda13/lib/python3.11/site-packages/optuna/trial/_trial.py:664: UserWarning: Fixed parameter min_sum_hessian_in_leaf with value 1e-06 is out of range for distribution FloatDistribution(high=0.1, log=True, low=0.0001, step=None).
  optuna_warn(


[I 2026-10-06 04:31:51,806] Trial 22 finished with value: 0.11926476068939489 and parameters: {'num_iterations': 712, 'num_leaves': 86, 'feature_fraction': 0.4, 'min_sum_hessian_in_leaf': 1e-06, 'learning_rate': 0.00757969257611117, 'min_data_in_leaf': 0, 'scale_pos_weight': 1, 'prob_corte': 0.006351497950491821}. Best is trial 11 with value: 0.1269863270962039.
[I 2026-10-06 04:33:12,420] Trial 23 finished with value: 0.13938145899476917 and parameters: {'num_iterations': 2140, 'num_leaves': 133, 'feature_fraction': 0.536030890639101, 'min_sum_hessian_in_leaf': 0.0001, 'learning_rate': 0.02, 'min_data_in_leaf': 0, 'scale_pos_weight': 1, 'prob_corte': 0.0047944533654560905}. Best is trial 23 with value: 0.13938145899476917.
[I 2026-10-06 04:34:33,829] Trial 24 finished with value: 0.13943876215731632 and parameters: {'num_iterations': 2140, 'num_leaves': 133, 'feature_fraction': 0.536030890639101, 'min_sum_hessian_in_leaf': 0.0001, 'learning_rate': 0.02, 'min_data_in_leaf': 0, 'scale_p

## Resultados de la optimización

In [33]:
tb_bayesiana = study.trials_dataframe()
tb_bayesiana["iter"] = np.arange(1, len(tb_bayesiana) + 1)
tb_bayesiana = (
    tb_bayesiana
    .sort_values("value", ascending=False, na_position="last")
    .reset_index(drop=True)
)
tb_bayesiana.to_csv(
    os.path.join(EXPERIMENT_URL, "BO_log.txt"),
    sep="	",
    index=False,
)

prob_corte_optimo = float(study.best_trial.params["prob_corte"])
# PARAM.yml conserva los HP de LightGBM; el corte queda en BO_log.txt y resumen.md.
PARAM["out"]["lgbm"]["mejores_hiperparametros"] = {
    nombre: valor for nombre, valor in study.best_trial.params.items()
    if nombre != "prob_corte"
}
PARAM["out"]["lgbm"]["y"] = float(study.best_value)

with open(os.path.join(EXPERIMENT_URL, "PARAM.yml"), "w", encoding="utf-8") as archivo:
    yaml.safe_dump(PARAM, archivo, sort_keys=False, allow_unicode=True)

print(PARAM["out"]["lgbm"]["mejores_hiperparametros"])
print("Mejor Average Precision:", PARAM["out"]["lgbm"]["y"])
tb_bayesiana.head()

{'num_iterations': 3668, 'num_leaves': 204, 'feature_fraction': 0.3847092908193044, 'min_sum_hessian_in_leaf': 0.00010000000000000009, 'learning_rate': 0.025, 'min_data_in_leaf': 0, 'scale_pos_weight': 1.0}
Mejor Average Precision: 0.146249656032765


,number,value,datetime_start,datetime_complete,duration,params_feature_fraction,params_learning_rate,params_min_data_in_leaf,params_min_sum_hessian_in_leaf,params_num_iterations,params_num_leaves,params_prob_corte,params_scale_pos_weight,system_attrs_fixed_params,system_attrs_gp:relative_params:0,state,iter
0,141,0.146250,2026-10-06 08:21:57.544136,2026-10-06 08:24:18.012466,0 days 00:02:20.468330,0.384709,0.025000,0,0.0001,3668,204,0.003545,1.000000,NaN,"{""feature_fraction"": 0.3847092908193044, ""lear...",COMPLETE,142
1,152,0.145880,2026-10-06 08:46:26.084876,2026-10-06 08:48:51.364112,0 days 00:02:25.279236,0.462775,0.025000,0,0.0001,3362,251,0.003545,1.000000,NaN,"{""feature_fraction"": 0.4627750965022953, ""lear...",COMPLETE,153
2,110,0.145262,2026-10-06 06:59:45.151364,2026-10-06 07:01:41.336560,0 days 00:01:56.185196,0.330960,0.025000,0,0.0001,3084,227,0.008646,0.754676,NaN,"{""feature_fraction"": 0.33095966855765835, ""lea...",COMPLETE,111
3,150,0.144967,2026-10-06 08:42:20.262415,2026-10-06 08:44:57.724248,0 days 00:02:37.461833,0.426132,0.013923,0,0.0001,3658,247,0.003545,1.000000,NaN,"{""feature_fraction"": 0.4261322704767676, ""lear...",COMPLETE,151
4,184,0.144960,2026-10-06 09:59:13.879514,2026-10-06 10:01:27.896578,0 days 00:02:14.017064,0.345427,0.025000,0,0.0001,3609,199,0.005957,1.000000,NaN,"{""feature_fraction"": 0.34542727310493715, ""lea...",COMPLETE,185


## Producción: marzo a julio para predecir agosto

In [34]:
dataset_final_cpu = (
    df_completo
    .loc[df_completo["foto_mes"].isin(PARAM["train_final"])]
    .to_pandas()
    .reset_index(drop=True)
)
etiquetas_julio = etiquetar_julio(prob_corte_optimo)
dataset_final_cpu.loc[
    dataset_final_cpu["foto_mes"].isin(meses_julio), "ternaria"
] = etiquetas_julio
dataset_final_cpu["clase01"] = (
    dataset_final_cpu["ternaria"].isin(["BAJA+1", "BAJA+2"])
    .astype(np.int8)
)

dataset_final_cpu.groupby(["foto_mes", "ternaria"]).size()

foto_mes  ternaria
202103    BAJA+1        1019
          BAJA+2         960
          continua    160921
202104    BAJA+1         964
          BAJA+2        1139
          continua    161181
202105    BAJA+1        1143
          BAJA+2         870
          continua    161755
202106    BAJA+1         874
          BAJA+2        1098
          continua    162142
202107    BAJA+1        1103
          BAJA+2       15000
          CONTINUA    148245
dtype: int64

In [35]:
param_final = {
    **PARAM["lgbm"]["param_fijos"],
    **PARAM["out"]["lgbm"]["mejores_hiperparametros"],
}

num_boost_round_final = int(param_final.pop("num_iterations"))
param_final["min_data_in_leaf"] = round(
    param_final["min_data_in_leaf"] / PARAM["trainingstrategy"]["undersampling"]
)
param_final

dtrain_final = lgb.Dataset(
    data=dataset_final_cpu[campos_buenos],
    label=dataset_final_cpu["clase01"],
    feature_name=campos_buenos,
)

modelo_final = lgb.train(
    params=param_final,
    train_set=dtrain_final,
    num_boost_round=num_boost_round_final,
)

In [36]:
arboles_df = modelo_final.trees_to_dataframe()
splits_df = arboles_df.loc[arboles_df["split_feature"].notna()]

tb_importancia = (
    splits_df
    .groupby("split_feature", as_index=False)
    .agg(
        Gain=("split_gain", "sum"),
        Cover=("count", "sum"),
        Frequency=("split_feature", "size"),
    )
    .rename(columns={"split_feature": "Feature"})
)

for columna in ["Gain", "Cover", "Frequency"]:
    total = tb_importancia[columna].sum()
    if total > 0:
        tb_importancia[columna] = tb_importancia[columna] / total

tb_importancia = tb_importancia.sort_values("Gain", ascending=False)
tb_importancia.to_csv(
    os.path.join(EXPERIMENT_URL, "impo.txt"),
    sep="\t",
    index=False,
)
modelo_final.save_model(os.path.join(EXPERIMENT_URL, "modelo.txt"))

tb_importancia.head(20)

,Feature,Gain,Cover,Frequency
302,foto_mes,0.069418,0.046140,0.003063
299,ctrx_quarter,0.058902,0.009284,0.009496
301,ctrx_quarter_lag_pond,0.055465,0.002695,0.004911
318,mcaja_ahorro,0.044606,0.008998,0.009406
389,mpayroll,0.042485,0.009676,0.002840
234,cpayroll_trx,0.026729,0.006138,0.001904
122,Visa_msaldopesos,0.023569,0.003968,0.005579
215,cliente_antiguedad_delta_lag_pond,0.023086,0.011692,0.005749
359,mcuentas_saldo,0.021195,0.016360,0.012123
421,mtarjeta_visa_consumo,0.020363,0.004470,0.004127


In [37]:
dfuture_cpu = (
    df_completo
    .loc[df_completo["foto_mes"].isin(PARAM["future"])]
    .to_pandas()
    .reset_index(drop=True)
)

prediccion = modelo_final.predict(dfuture_cpu[campos_buenos])

tb_prediccion = dfuture_cpu[["numero_de_cliente", "foto_mes", "ternaria"]].copy()
tb_prediccion["prob"] = prediccion
tb_prediccion[["numero_de_cliente", "foto_mes", "prob"]].to_csv(
    os.path.join(EXPERIMENT_URL, "prediccion.txt"),
    sep="\t",
    index=False,
)

tb_prediccion.head()

,numero_de_cliente,foto_mes,ternaria,prob
0,12159854,202108,NaN,8.724335e-06
1,12159858,202108,NaN,3.630242e-06
2,12160484,202108,NaN,1.276281e-07
3,12160591,202108,NaN,1.425858e-08
4,12160747,202108,NaN,6.819066e-07


In [38]:
with open(os.path.join(EXPERIMENT_URL, "PARAM.yml"), "w", encoding="utf-8") as archivo:
    yaml.safe_dump(PARAM, archivo, sort_keys=False, allow_unicode=True)

## Documentación de la segunda BO

In [39]:
clase_julio = np.isin(etiquetas_julio, ["BAJA+1", "BAJA+2"])
training_julio = (
    (azar_julio <= PARAM["trainingstrategy"]["undersampling"])
    | clase_julio
)
contenido_md = [
    f"# Resumen — {experiment_name}",
    "",
    "- Etiquetas: reales en marzo a junio y BAJA+1 de julio; BAJA+2 estimadas en julio.",
    "- Julio participa solamente en el entrenamiento, no en la validación.",
    f"- Meses de validación: {', '.join(map(str, meses_reales))}",
    "- Validación: se conserva dvalidate de la primera BO, con etiquetas reales.",
    f"- Experimento que generó las probabilidades de julio: `{experimento_origen}`",
    "- Probabilidades por cliente y mes: `predicciones_julio/prediccion.txt`.",
    "- Estas probabilidades corresponden a la clase conjunta BAJA+1/BAJA+2 del primer modelo.",
    f"- Meses con BAJA+2 estimadas: {', '.join(map(str, meses_julio))}",
    "- Regla: conservar BAJA+1 real; asignar BAJA+2 si prob >= prob_corte; CONTINUA en el resto.",
    f"- Puestos para definir el rango: {min(PARAM['cortes'])} a {max(PARAM['cortes'])}, excluyendo BAJA+1 conocidas.",
    f"- Rango de prob_corte: {rangos_bo['prob_corte']}",
    f"- Mejor prob_corte: {prob_corte_optimo}",
    f"- BAJA+1 reales conservadas en julio: {int(baja1_julio.sum())}",
    f"- BAJA+2 estimadas en julio: {int((etiquetas_julio == 'BAJA+2').sum())}",
    f"- Registros de entrenamiento BO con el mejor corte: {len(X_train_bo) + int(training_julio.sum())}",
    f"- Registros de validación BO: {len(X_validate_bo)}",
    f"- Registros de entrenamiento final: {len(dataset_final_cpu)}",
    f"- Registros predichos: {len(dfuture_cpu)}",
    "- Agosto no tiene BAJA+2 real: no se calcula ganancia sobre ese mes.",
    f"- Dataset: `{dataset_file}`",
    f"- Meses de entrenamiento: {', '.join(map(str, PARAM['train']))}",
    f"- Meses de entrenamiento final: {', '.join(map(str, PARAM['train_final']))}",
    f"- Meses a predecir: {', '.join(map(str, PARAM['future']))}",
    f"- Semilla: {PARAM['semilla_primigenia']}",
    "- Clase positiva: BAJA+1 y BAJA+2.",
    f"- Columnas excluidas: {', '.join(sorted(columnas_excluidas))}",
    f"- Porcentaje para entrenamiento en el split: {PARAM['training_pct']}",
    f"- Undersampling: {PARAM['trainingstrategy']['undersampling']}",
    f"- Corridas por BO: {PARAM['hyperparametertuning']['iteraciones']}",
    f"- Inicializaciones totales: {PARAM['hyperparametertuning']['n_startup_trials']}",
    f"- Inicializaciones desde RDS: {len(puntos_iniciales_bo)}",
    f"- Inicializaciones aleatorias: {PARAM['hyperparametertuning']['n_startup_trials'] - len(puntos_iniciales_bo)}",
    f"- Fuente de las inicializaciones: `{archivo_validaciones}`",
    f"- IDs de validación usados: {', '.join(map(str, modelos_iniciales['id_validacion']))}",
    "- Selección: mayor ganancia dentro del dataset actual; se completa con los demás datasets.",
    "- min_data_in_leaf de los RDS: se revierte el ajuste por undersampling antes de inicializar.",
    f"- Rangos de HP: `{rangos_bo}`",
    f"- Mejor trial de Optuna: {study.best_trial.number}",
    f"- Mejor Average Precision: {PARAM['out']['lgbm']['y']}",
    f"- Mejores HP: `{PARAM['out']['lgbm']['mejores_hiperparametros']}`",
]
with open(os.path.join(EXPERIMENT_URL, "resumen.md"), "w", encoding="utf-8") as archivo:
    archivo.write("\n".join(contenido_md) + "\n")